In [2]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

tqdm.pandas()

In [3]:
df = pd.read_csv("/content/customer_support_tickets.csv")

In [4]:
print(df.columns.tolist())
print(df.head())

['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score']
    Ticket_ID     Customer_Name              Customer_Email  \
0  TKT-100000      George Simon  lisastrickland@example.com   
1  TKT-100001    Scott Thompson          wevans@example.org   
2  TKT-100002    Jennifer Smith        oleonard@example.net   
3  TKT-100003    Rachel Bullock     katherine67@example.net   
4  TKT-100004  Thomas Parks DDS       raykelsey@example.com   

                    Ticket_Subject  \
0  Hours of operation - Individual   
1          Data not syncing - Card   
2            2FA issues - Question   
3               Login failed - Let   
4        Refund status - Attention   

                                  Ticket_Description   Issue_Category  \
0  Hi Support, Where is your headquarters located...  General Inquiry   
1  Hi Support, The 

In [5]:
text_cols = ['Ticket_Subject', 'Ticket_Description']

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

df['Priority_Level'] = df['Priority_Level'].fillna("Low")

df['Resolution_Time_Hours'] = (
    pd.to_numeric(df['Resolution_Time_Hours'],
                  errors='coerce')
    .fillna(df['Resolution_Time_Hours'].median())
)

#SIGNAL 1 : Keyword Severity

In [6]:
critical_keywords = [
    "security breach",
    "data breach",
    "fraud",
    "payment failure",
    "account hacked",
    "account locked",
    "system crash",
    "server down",
    "critical failure",
    "unauthorized access",
    "ransomware"
]

high_keywords = [
    "error",
    "unable",
    "failed",
    "not working",
    "cannot login",
    "bug",
    "service unavailable",
    "payment issue",
    "access denied",
    "outage"
]

medium_keywords = [
    "slow",
    "delay",
    "issue",
    "problem",
    "incorrect",
    "missing",
    "warning"
]

Combining Ticket Text

In [7]:
df['full_text'] = (
    df['Ticket_Subject']
    + " "
    + df['Ticket_Description']
).str.lower()

Keyword Severity Function

In [8]:
def keyword_severity(text):

    critical_score = sum(
        keyword in text
        for keyword in critical_keywords
    )

    high_score = sum(
        keyword in text
        for keyword in high_keywords
    )

    medium_score = sum(
        keyword in text
        for keyword in medium_keywords
    )

    if critical_score > 0:
        return 4

    elif high_score >= 2:
        return 3

    elif high_score >= 1:
        return 2

    elif medium_score >= 1:
        return 2

    else:
        return 1

In [9]:
df['keyword_severity'] = df['full_text'].progress_apply(
    keyword_severity
)

100%|██████████| 20000/20000 [00:00<00:00, 134689.25it/s]


#SIGNAL 2 : Resolution-Time Severity

In [10]:
def resolution_severity(hours):

    if hours >= 72:
        return 4

    elif hours >= 48:
        return 3

    elif hours >= 24:
        return 2

    else:
        return 1

In [11]:
df['resolution_severity'] = (
    df['Resolution_Time_Hours']
    .apply(resolution_severity)
)

#SIGNAL FUSION

Weighted combination

In [12]:
df['severity_score'] = (
    0.7 * df['keyword_severity']
    +
    0.3 * df['resolution_severity']
)

Convert to Final Severity

In [13]:
def final_severity(score):

    if score >= 3.5:
        return "Critical"

    elif score >= 2.5:
        return "High"

    elif score >= 1.5:
        return "Medium"

    else:
        return "Low"

In [14]:
df['inferred_severity'] = (
    df['severity_score']
    .apply(final_severity)
)

#ASSIGNED PRIORITY

Normalize labels

In [15]:
def normalize_priority(priority):

    priority = str(priority).strip().lower()

    mapping = {
        "low": "Low",
        "medium": "Medium",
        "high": "High",
        "critical": "Critical"
    }

    return mapping.get(priority, "Low")

In [16]:
df['assigned_priority'] = (
    df['Priority_Level']
    .apply(normalize_priority)
)

#PSEUDO LABEL GENERATION

In [17]:
df['pseudo_label'] = (
    df['assigned_priority']
    !=
    df['inferred_severity']
).astype(int)

#MISMATCH TYPE

In [18]:
priority_map = {
    "Low":1,
    "Medium":2,
    "High":3,
    "Critical":4
}

In [19]:
def mismatch_type(row):

    assigned = priority_map[row['assigned_priority']]
    inferred = priority_map[row['inferred_severity']]

    if assigned < inferred:
        return "Hidden Crisis"

    elif assigned > inferred:
        return "False Alarm"

    return "Consistent"

In [20]:
df['mismatch_type'] = df.apply(
    mismatch_type,
    axis=1
)

#Quick Analysis

In [21]:
print("\nPseudo Label Distribution\n")

print(
    df['pseudo_label']
    .value_counts(normalize=True)
    * 100
)


Pseudo Label Distribution

pseudo_label
1    61.755
0    38.245
Name: proportion, dtype: float64


Check Examples

In [22]:
df[
    [
        'Ticket_Subject',
        'assigned_priority',
        'inferred_severity',
        'pseudo_label',
        'mismatch_type'
    ]
].sample(10)

,Ticket_Subject,assigned_priority,inferred_severity,pseudo_label,mismatch_type
12435,Stolen card - Draw,Critical,Low,1,False Alarm
17381,Data not syncing - Include,Medium,Low,1,False Alarm
9649,Refund status - Art,Low,Medium,1,Hidden Crisis
1292,Alert notification - Necessary,Critical,Low,1,False Alarm
4934,Update credit card - Soon,Medium,Low,1,False Alarm
16242,Charged twice - Down,Low,Medium,1,Hidden Crisis
4151,Charged twice - Approach,Medium,Medium,0,Consistent
568,Refund status - Play,High,Low,1,False Alarm
8123,Installation issue - Particular,High,Medium,1,False Alarm
13228,Pricing tiers - Thing,Low,Low,0,Consistent


Save Dataset

In [23]:
df.to_csv(
    "pseudo_labeled_tickets.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


#STAGE 2

In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

from datasets import Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [25]:
df = pd.read_csv("pseudo_labeled_tickets.csv")

In [26]:
print(df.shape)
print(df['pseudo_label'].value_counts())

(20000, 20)
pseudo_label
1    12351
0     7649
Name: count, dtype: int64


#Creating Model Input Text

In [27]:
df['model_text'] = (
    "Subject: " + df['Ticket_Subject'].astype(str)
    + " Description: " + df['Ticket_Description'].astype(str)
    + " Channel: " + df['Ticket_Channel'].astype(str)
    + " Category: " + df['Issue_Category'].astype(str)
)

Keep Only Required Columns

In [28]:
df = df[['model_text','pseudo_label']]

In [29]:
df.columns = ['text','label']

In [30]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

Converting to HuggingFace Dataset

In [31]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

#tokenizing

In [32]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [33]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [34]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

#Loading DistilBERT

In [35]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Metrics

In [36]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    acc = accuracy_score(
        labels,
        predictions
    )

    f1 = f1_score(
        labels,
        predictions
    )

    return {
        "accuracy": acc,
        "f1": f1
    }

#Training Arguments

In [37]:
import transformers
print(transformers.__version__)

5.10.2


In [38]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none"
)

#TRAINER

In [39]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [40]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.637839,0.646466,0.617500,0.763524
2,0.644732,0.632938,0.617500,0.763524
3,0.630739,0.633671,0.617500,0.763524


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.6512855962117513, metrics={'train_runtime': 1238.1592, 'train_samples_per_second': 38.767, 'train_steps_per_second': 2.423, 'total_flos': 3179217567744000.0, 'train_loss': 0.6512855962117513, 'epoch': 3.0})

#Evaluate

In [41]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.630739,0.632938,3,0.617500,0.763524


{'eval_loss': 0.6329378485679626, 'eval_accuracy': 0.6175, 'eval_f1': 0.7635239567233385}


#Detailed Report

In [42]:
preds = trainer.predict(test_dataset)

y_pred = np.argmax(
    preds.predictions,
    axis=1
)

y_true = preds.label_ids

print(
    classification_report(
        y_true,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1530
           1       0.62      1.00      0.76      2470

    accuracy                           0.62      4000
   macro avg       0.31      0.50      0.38      4000
weighted avg       0.38      0.62      0.47      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#Save Model

In [43]:
model.save_pretrained(
    "saved_model"
)

tokenizer.save_pretrained(
    "saved_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_model/tokenizer_config.json', 'saved_model/tokenizer.json')

#READ.ME

In [44]:
results = trainer.evaluate()

print("Accuracy :", results['eval_accuracy'])
print("F1 Score :", results['eval_f1'])

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.630739,0.632938,3,0.617500,0.763524


Accuracy : 0.6175
F1 Score : 0.7635239567233385


#Evidence Dossier Generation

In [45]:
import pandas as pd

df = pd.read_csv("pseudo_labeled_tickets.csv")

##Priority Mapping

In [46]:
priority_map = {
    "Low":1,
    "Medium":2,
    "High":3,
    "Critical":4
}

#Extract Keyword Evidence

In [47]:
critical_keywords = [
    "crash",
    "security",
    "breach",
    "fraud",
    "hacked",
    "locked",
    "server down",
    "payment failed"
]

high_keywords = [
    "error",
    "failed",
    "unable",
    "not working",
    "outage",
    "access denied"
]

In [48]:
def get_keyword_evidence(text):

    text = str(text).lower()

    evidence = []

    for word in critical_keywords:

        if word in text:

            evidence.append({
                "signal":"keyword",
                "value":word,
                "weight":"high"
            })

    for word in high_keywords:

        if word in text:

            evidence.append({
                "signal":"keyword",
                "value":word,
                "weight":"medium"
            })

    return evidence

#Resolution Time Evidence

In [49]:
def resolution_evidence(hours):

    if hours >= 72:

        return {
            "signal":"resolution_time",
            "value":f"{hours:.1f} hours",
            "interpretation":"Very high resolution time"
        }

    elif hours >= 48:

        return {
            "signal":"resolution_time",
            "value":f"{hours:.1f} hours",
            "interpretation":"High resolution time"
        }

    return {
        "signal":"resolution_time",
        "value":f"{hours:.1f} hours",
        "interpretation":"Normal resolution time"
    }

#Determine Mismatch Type

In [50]:
def get_mismatch_type(row):

    assigned = priority_map[
        row["assigned_priority"]
    ]

    inferred = priority_map[
        row["inferred_severity"]
    ]

    if assigned < inferred:
        return "Hidden Crisis"

    elif assigned > inferred:
        return "False Alarm"

    return "Consistent"

#Constraint Analysis

In [51]:
def generate_analysis(row):

    assigned = row["assigned_priority"]

    inferred = row["inferred_severity"]

    mismatch = row["mismatch_type"]

    if mismatch == "Hidden Crisis":

        return (
            f"The ticket was assigned {assigned} "
            f"but evidence indicates {inferred}. "
            f"Keywords and resolution patterns "
            f"suggest underestimated severity."
        )

    elif mismatch == "False Alarm":

        return (
            f"The ticket was assigned {assigned} "
            f"but evidence indicates {inferred}. "
            f"The priority appears overestimated."
        )

    return "Priority assignment appears consistent."

#Confidence Score

In [52]:
def confidence_score(row):

    assigned = priority_map[
        row["assigned_priority"]
    ]

    inferred = priority_map[
        row["inferred_severity"]
    ]

    delta = abs(
        inferred - assigned
    )

    confidence = min(
        0.6 + 0.1*delta,
        0.99
    )

    return round(confidence,2)

#Generate Dossier

In [53]:
import json

In [54]:
def generate_dossier(row):

    text = str(row["full_text"])

    keyword_ev = get_keyword_evidence(text)

    resolution_ev = resolution_evidence(
        row["Resolution_Time_Hours"]
    )

    evidence = keyword_ev

    evidence.append(resolution_ev)

    dossier = {

        "ticket_id": row["Ticket_ID"],

        "assigned_priority":
            row["assigned_priority"],

        "inferred_severity":
            row["inferred_severity"],

        "mismatch_type":
            row["mismatch_type"],

        "feature_evidence":
            evidence,

        "constraint_analysis":
            generate_analysis(row),

        "confidence":
            confidence_score(row)
    }

    return dossier

#Generate Only For Mismatches

In [55]:
mismatch_df = df[
    df["pseudo_label"] == 1
]

In [56]:
dossiers = []

for _, row in mismatch_df.iterrows():

    dossiers.append(
        generate_dossier(row)
    )

#Save JSON File

In [57]:
with open(
    "evidence_dossiers.json",
    "w"
) as f:

    json.dump(
        dossiers,
        f,
        indent=4
    )

In [58]:
print(
    json.dumps(
        dossiers[0],
        indent=4
    )
)

{
    "ticket_id": "TKT-100000",
    "assigned_priority": "High",
    "inferred_severity": "Low",
    "mismatch_type": "False Alarm",
    "feature_evidence": [
        {
            "signal": "resolution_time",
            "value": "43.0 hours",
            "interpretation": "Normal resolution time"
        }
    ],
    "constraint_analysis": "The ticket was assigned High but evidence indicates Low. The priority appears overestimated.",
    "confidence": 0.8
}


#STREAMLIT

In [16]:
%%writefile app.py

import streamlit as st

st.title("Support Integrity Auditor")
st.write("Project Demo")

Overwriting app.py


In [17]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇
up to date, audited 23 packages in 931ms
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠇

In [21]:
import pandas as pd

df = pd.read_csv("pseudo_labeled_tickets.csv")
print(df.columns.tolist())

['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score', 'full_text', 'keyword_severity', 'resolution_severity', 'severity_score', 'inferred_severity', 'assigned_priority', 'pseudo_label', 'mismatch_type']


In [24]:
import pandas as pd

df = pd.read_csv("pseudo_labeled_tickets.csv")

print(df.columns.tolist())

['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject', 'Ticket_Description', 'Issue_Category', 'Priority_Level', 'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours', 'Assigned_Agent', 'Satisfaction_Score', 'full_text', 'keyword_severity', 'resolution_severity', 'severity_score', 'inferred_severity', 'assigned_priority', 'pseudo_label', 'mismatch_type']


In [25]:
!pip install streamlit -q
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
up to date, audited 23 packages in 2s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠴

In [26]:
!streamlit run app.py > logs.txt 2>&1 &

In [ ]:
!npx localtunnel --port 8501

⠙⠹your url is: https://lovely-berries-hope.loca.lt


In [28]:
%%writefile train_pipeline.py

print("Training Pipeline")

Writing train_pipeline.py


In [29]:
%%writefile predict.py

print("Inference Script")

Writing predict.py


In [30]:
%%writefile README.md

# Support Integrity Auditor (SIA)

AI-powered system for detecting priority mismatches in CRM support tickets.

## Features
- Pseudo-label generation
- Mismatch detection
- Evidence dossier generation
- Streamlit dashboard

## Run

pip install -r requirements.txt

streamlit run app.py

Writing README.md


In [31]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn
transformers
torch

Writing requirements.txt


In [32]:
!ls /content

app.py				    predict.py
customer_support_tickets.csv	    pseudo_labeled_tickets.csv
drive				    README.md
enhanced_customer_support_data.csv  requirements.txt
evidence_dossiers.json		    results
logs.txt			    sample_data
node_modules			    saved_model
package.json			    train_pipeline.py
package-lock.json


In [34]:
from google.colab import files

files.download("app.py")
files.download("train_pipeline.py")
files.download("predict.py")
files.download("README.md")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>